In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pandas as pd

#Navigator settings
options = webdriver.ChromeOptions()

options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                     "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

#Pour lancer Chrome
driver = webdriver.Chrome(options=options)

try:
    driver.get("https://ieeexplore.ieee.org/Xplore/home.jsp")

    #Mise en place d'un timeout si erreur de recherche
    wait = WebDriverWait(driver, 20)

    #Attendre que la barre de recherche soit présente
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "global-search-bar")))

    #Mise en place d'un try except pour le gestion des erreurs
    #Chercher l'input à l'intérieur de la barre de recherche qui est souvent global-search-bar-input
    try:
        search_input = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".global-search-bar input"))
        )
    except TimeoutException:
        # fallback : chercher un input de type search ou avec placeholder
        search_input = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "input[type='search'], input[placeholder*='Search']"))
        )

    #petit délai optionnel pour s'assurer que l'input est prêt
    time.sleep(0.5)

    #définition et envoie de la requête
    query = "reinforcement learning for trading"
    search_input.clear()
    search_input.send_keys(query)

    #soumission de requête
    search_input.send_keys(Keys.ENTER)
    #attendre que les résultats apparaissent
    try:
        wait_results = WebDriverWait(driver, 20)
        wait_results.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".List-results, .results-container, .SearchResults"))) 
        print("Recherche lancée — résultats chargés (ou en cours).")
    except TimeoutException:
        print("Recherche lancée, mais les résultats n'ont pas chargé dans le délai imparti.")

except Exception as e:
    print("Erreur durant l'exécution :", e)

#Cliquer sur le bouton accepter les cookies si présent
try:
    accept_button = WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(.,'Accept') or contains(.,'Accepter')]"))
    )
    accept_button.click()
except Exception:
    pass  # pas de cookies alors on passe


Recherche lancée — résultats chargés (ou en cours).


In [ ]:
#On recherche les liens présents dans la page
links = driver.find_elements(By.TAG_NAME, "a")

#On initialise une liste titres et liens qui contiendrons les informations éponymes
titres=[]
liens=[]

suivant=True
#Si bouton '>' est présent présent, alors on clique dessus et on extrait les articles des pages suivantes
while suivant==True:

    #Récupérer tous les éléments <a> à savoir les liens
    links = driver.find_elements(By.TAG_NAME, "a")

    # Filtrer ceux qui ont un texte non vide (souvent les titres)
    for link in links:
        text = link.text.strip()
        href = link.get_attribute("href")
        #Définition des conditions d'inclusion des articles
        if (text and href) and ("reinforcement learning" in text.lower() and "trading" in text.lower()) :
            print(f"{text} -> {href}")
            titres.append(text)
            liens.append(href)
    #On essaie de cliquer sur le bouton '>' si présent
    try:
        next_btn = driver.find_element(By.CLASS_NAME, "next-btn")
        next_btn.click()
        time.sleep(2)
    #Sinon, cela signifie que le scrapping est fini
    except:
            print("Fin des pages : 'Next' n'est plus un lien.")
            suivant=False


Equilibrium Analysis for Electricity Market Considering Carbon Emission Trading Based on Multi-Agent Deep Reinforcement Learning -> https://ieeexplore.ieee.org/document/10294544/
Cryptosentiment: A Dataset and Baseline for Sentiment-Aware Deep Reinforcement Learning for Financial Trading -> https://ieeexplore.ieee.org/document/10193330/
Application of Deep Reinforcement Learning in Financial Quantitative Trading -> https://ieeexplore.ieee.org/document/9851105/
Interactive Learning - Implementation of ChatGPT and Reinforcement Learning in Local Energy Trading -> https://ieeexplore.ieee.org/document/10807616/
Optimizing Deep Reinforcement Learning Models for Stock Trading with Technical Indicators -> https://ieeexplore.ieee.org/document/10949548/
Improving Deep Reinforcement Learning for Financial Trading Using Neural Network Distillation -> https://ieeexplore.ieee.org/document/9231849/
High-Frequency Trading Algorithm Optimization Based on Deep Reinforcement Learning and Generative Adve

In [ ]:
#On ajoute tous les articles conservés dans un CSV
df=pd.DataFrame({"Titres":titres,"Liens":liens})
df.to_csv("IEEE_Scrapping.csv",sep=";",index=False)

In [95]:
df["Liens"]

0      https://ieeexplore.ieee.org/document/10294544/
1      https://ieeexplore.ieee.org/document/10193330/
2       https://ieeexplore.ieee.org/document/9851105/
3      https://ieeexplore.ieee.org/document/10807616/
4      https://ieeexplore.ieee.org/document/10949548/
                            ...                      
140    https://ieeexplore.ieee.org/document/10966432/
141    https://ieeexplore.ieee.org/document/10444990/
142     https://ieeexplore.ieee.org/document/9596598/
143     https://ieeexplore.ieee.org/document/9877940/
144    https://ieeexplore.ieee.org/document/10086597/
Name: Liens, Length: 145, dtype: object

In [ ]:
#Code pour récupérer les abstracts des articles
driver = webdriver.Chrome(options=options)
#On initialise un liste qui va contenir les abstracts
abstracts=[]
for url in df["Liens"]:
    time.sleep(3)
    driver.get(url)
    try:
        abstract = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "abstract-text")))
        abstracts.append(abstract.text.replace("Abstract:\n",""))
        print(abstract.text)
    except:
        abstracts.append("NA")
        

Abstract:
With the proposal of China's carbon peaking and carbon neutrality goals, carbon emission trading (CET) is gradually participating in the electricity market to accelerate carbon emission reduction and the improvement of power supply structure. In this study, we analyze the impacts of CET on the electricity market based on the electricity market equilibrium model and multi-agent deep reinforcement learning (MADL) method. We firstly establish the electricity market clearing process involved CET and develop the bi-level problem to model the electricity market equilibrium with strategic generation company (GENCO) bidders. Then, a multi-agent Twin Delayed Deep Deterministic Policy Gradient (MATD3) algorithm is applied to solve the market equilibrium described above. Finally, we simulate multiple cases based on a modified IEEE 30-bus system. The result shows that an excessive carbon price can raise the nodal electricity price and have a negative influence on reducing carbon emission

In [100]:
len(abstracts)

145

In [101]:
df.head()
df["Abstracts"]=abstracts
df.head()

,Titres,Liens,Abstracts
0,Equilibrium Analysis for Electricity Market Co...,https://ieeexplore.ieee.org/document/10294544/,With the proposal of China's carbon peaking an...
1,Cryptosentiment: A Dataset and Baseline for Se...,https://ieeexplore.ieee.org/document/10193330/,Deep Learning (DL) models have been applied in...
2,Application of Deep Reinforcement Learning in ...,https://ieeexplore.ieee.org/document/9851105/,Quantitative trading refers to the use of comp...
3,Interactive Learning - Implementation of ChatG...,https://ieeexplore.ieee.org/document/10807616/,"Within the Local Electricity Market (LEM), Dis..."
4,Optimizing Deep Reinforcement Learning Models ...,https://ieeexplore.ieee.org/document/10949548/,This paper presents the development and optimi...


In [85]:
df.shape

(24, 3)

In [ ]:
#On remplace le fichier IEEE Scrapping de base pas un nouveau qui contient cette fois-ci les titres, liens et abstracts des articles 
df.to_csv("IEEE_Scrapping.csv",sep=";",index=False)